1. Quais países mais ganharam ou perderam espaço relativo (market share em Valor FOB) entre 1997 e o período recente?



In [0]:
%sql
USE CATALOG mvp;
USE SCHEMA gold;

WITH importacoes_pais_geral AS (
    SELECT 
        co_ano,
        co_pais,
        SUM(vl_fob_pais_setor) AS vl_fob_pais_ano
    FROM gold_market_share_setor_pais
    GROUP BY co_ano, co_pais
),
total_brasil_ano AS (
    SELECT 
        co_ano,
        SUM(vl_fob_pais_setor) AS vl_fob_total_brasil
    FROM gold_market_share_setor_pais
    GROUP BY co_ano
),
market_share_global AS (
    SELECT 
        p.co_ano,
        p.co_pais,
        (p.vl_fob_pais_ano / t.vl_fob_total_brasil) * 100 AS ms_global_perc
    FROM importacoes_pais_geral p
    JOIN total_brasil_ano t ON p.co_ano = t.co_ano
),
inicio AS (
    SELECT co_pais, ms_global_perc AS ms_1997
    FROM market_share_global
    WHERE co_ano = 1997
),
fim AS (
    SELECT co_pais, ms_global_perc AS ms_recente, co_ano AS ano_recente
    FROM market_share_global
    WHERE co_ano = (SELECT MAX(co_ano) FROM market_share_global)
)
SELECT 
    f.ano_recente,
    i.co_pais,
    ROUND(i.ms_1997, 2) AS market_share_1997_perc,
    ROUND(f.ms_recente, 2) AS market_share_recente_perc,
    ROUND((f.ms_recente - i.ms_1997), 2) AS variacao_pontos_percentuais
FROM inicio i
JOIN fim f ON i.co_pais = f.co_pais
ORDER BY variacao_pontos_percentuais DESC;

2. Como essa realocação se distribui por grandes setores da economia (Capítulos NCM)?


In [0]:
%sql
USE CATALOG mvp;
USE SCHEMA gold;

WITH ranked_1997 AS (
    SELECT 
        setor_capitulo,
        co_pais AS pais_1997,
        market_share_perc AS ms_1997,
        ROW_NUMBER() OVER(PARTITION BY setor_capitulo ORDER BY market_share_perc DESC) as rn
    FROM gold_market_share_setor_pais
    WHERE co_ano = 1997
),
ranked_recente AS (
    SELECT 
        setor_capitulo,
        co_pais AS pais_recente,
        market_share_perc AS ms_recente,
        ROW_NUMBER() OVER(PARTITION BY setor_capitulo ORDER BY market_share_perc DESC) as rn
    FROM gold_market_share_setor_pais
    WHERE co_ano = (SELECT MAX(co_ano) FROM gold_market_share_setor_pais)
)
SELECT 
    r97.setor_capitulo,
    r97.pais_1997 AS maior_fornecedor_1997,
    ROUND(r97.ms_1997, 2) AS ms_1997_perc,
    rr.pais_recente AS maior_fornecedor_recente,
    ROUND(rr.ms_recente, 2) AS ms_recente_perc
FROM ranked_1997 r97
JOIN ranked_recente rr 
  ON r97.setor_capitulo = rr.setor_capitulo 
  AND r97.rn = 1 AND rr.rn = 1
ORDER BY r97.setor_capitulo;

3. Quais classes de produtos apresentaram maior volatilidade ou mudança estrutural na origem?

In [0]:
%sql
USE CATALOG mvp;
USE SCHEMA gold;

SELECT 
    setor_capitulo,
    MAX(market_share_perc) AS max_ms_historico,
    MIN(market_share_perc) AS min_ms_historico,
    ROUND((MAX(market_share_perc) - MIN(market_share_perc)), 2) AS amplitude_volatilidade_lider
FROM gold_market_share_setor_pais
GROUP BY setor_capitulo
ORDER BY amplitude_volatilidade_lider DESC
LIMIT 15;